# 01 — Data Prep & Quality Checks

**Business question this repo answers:** where is this subscription business losing revenue,
which levers actually retain subscribers, and what should the company do about it? This is a
retention and pricing analysis, not a churn-prediction leaderboard exercise — the predictive
model in notebook 05 is one section near the end, not the point.

**Scope:** `transactions.csv` (v1, full 2015-01-01 to 2017-02-28 history) combined with
`transactions_v2.csv` (v2, the supplement extending coverage to 2017-03-31) — together the full
per-user transaction ledger — plus `members_v3.csv` and `train_v2.csv`. `user_logs_v2.csv`
(30GB+ of raw listening events) is deliberately excluded — it's not needed for a
billing/retention analysis and would make this repo impossible to run on a laptop. See the
README's Data section for why v2 alone isn't enough: it's a thin, label-window-concentrated
supplement, not a full history on its own.

**Data boundaries** (see main README for the full statement):
- Coverage is roughly January 2015 to March 2017.
- Training labels (`train_v2.csv`) cover subscriptions expiring February 2017.
- Churn is defined as: no renewal within 30 days of membership expiry.
- All prices are New Taiwan dollars (NT$).

This notebook: loads the three files, checks them for the data-quality issues this dataset is
known to have, reconstructs per-user membership periods (the base unit every later notebook
builds on), and caches the result.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src import data_load, cohorts, plotting

plotting.set_style()
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [2]:
transactions = data_load.load_transactions()
members = data_load.load_members()
train = data_load.load_train()

print(f"transactions (v1+v2): {len(transactions):,} rows, {transactions['msno'].nunique():,} unique users")
print(f"members_v3:           {len(members):,} rows")
print(f"train_v2:             {len(train):,} rows, churn rate {train['is_churn'].mean():.4f}")

transactions (v1+v2): 22,975,416 rows, 2,426,143 unique users
members_v3:           6,769,473 rows
train_v2:             970,960 rows, churn rate 0.0899


### Coverage check
Verify the stated Jan 2015 – Mar 2017 boundary against the actual data.

In [3]:
print("transaction_date range:        ", transactions["transaction_date"].min().date(), "->", transactions["transaction_date"].max().date())
print("membership_expire_date range:  ", transactions["membership_expire_date"].min().date(), "->", transactions["membership_expire_date"].max().date())
print("registration_init_time range:  ", members["registration_init_time"].min().date(), "->", members["registration_init_time"].max().date())

transaction_date range:         2015-01-01 -> 2017-03-31
membership_expire_date range:   1970-01-01 -> 2036-10-15
registration_init_time range:   2004-03-26 -> 2017-04-29


### Data quality: members_v3

`bd` (age) is self-reported and known to contain implausible values in this dataset — zeros,
negative numbers, and triple-digit ages. We flag these rather than silently dropping or
imputing them, since segment-level analysis later needs to visibly carve out an "unknown age"
bucket rather than quietly biasing the sample.

In [4]:
implausible_age = ~members["bd"].between(10, 90)
print(f"{implausible_age.mean():.2%} of members have bd outside a plausible 10-90 range")
print(members.loc[implausible_age, "bd"].value_counts().head(10))

print("\ngender missing rate:", round(members["gender"].isna().mean(), 4))
print(members["registered_via"].value_counts(normalize=True).round(3))

67.20% of members have bd outside a plausible 10-90 range
bd
0      4540215
112       1309
106        743
103        449
117        439
102        424
95         373
94         330
97         323
104        307
Name: count, dtype: int64

gender missing rate: 0.6543


registered_via
 4    0.413
 3    0.243
 9    0.219
 7    0.119
 11   0.004
 13   0.001
 8    0.001
 5    0.000
 17   0.000
 2    0.000
 6    0.000
 19   0.000
 16   0.000
 14   0.000
 1    0.000
 10   0.000
 18   0.000
-1    0.000
Name: proportion, dtype: float64


We keep the raw `bd` column as-is (no imputation) and add an explicit `age_valid` flag downstream instead of guessing at true ages.

### Data quality: transactions (v1 + v2 combined)

In [5]:
dupe_rows = transactions.duplicated().sum()
print(f"exact duplicate rows: {dupe_rows:,}")

zero_price = (transactions["plan_list_price"] == 0).mean()
print(f"plan_list_price == 0: {zero_price:.4%} of rows")

overpaid = (transactions["actual_amount_paid"] > transactions["plan_list_price"]).mean()
print(f"actual_amount_paid > plan_list_price: {overpaid:.4%} of rows")

print(transactions["payment_plan_days"].value_counts(normalize=True).round(3).head(10))
print(transactions["payment_method_id"].nunique(), "distinct payment methods")

exact duplicate rows: 0
plan_list_price == 0: 6.6024% of rows
actual_amount_paid > plan_list_price: 3.7418% of rows
payment_plan_days
30    0.878
0     0.038
31    0.033
7     0.026
410   0.007
195   0.006
180   0.003
10    0.002
90    0.001
100   0.001
Name: proportion, dtype: float64
40 distinct payment methods


`plan_list_price == 0` rows are real (promotional/trial transactions) and are kept, but excluded from discount-depth calculations in `src/pricing.py` — a discount percentage off a $0 list price is undefined, not zero.

In [6]:
implausible_expiry = (transactions["membership_expire_date"] < "2015-01-01") | (
    transactions["membership_expire_date"] > "2020-01-01"
)
print(f"membership_expire_date outside a plausible 2015-2020 range: {implausible_expiry.sum():,} rows ({implausible_expiry.mean():.4%})")
print(transactions.loc[implausible_expiry, "membership_expire_date"].dt.year.value_counts().sort_index().head(10))

membership_expire_date outside a plausible 2015-2020 range: 12,688 rows (0.0552%)
membership_expire_date
1970    1774
1999       2
2005       1
2007       3
2008       1
2009       2
2010       3
2012      16
2013    3155
2014    4760
Name: count, dtype: int64


A small number of rows (well under 0.1%) have a `membership_expire_date` before 2015 — including
a cluster at exactly 1970-01-01, a classic null/epoch sentinel value rather than a real date — or
years past any plan length sold in this data (up to 2036). These are flagged here, not filtered:
at this scale they can't move any aggregate statistic downstream, but a reader auditing the
pipeline should be able to find them rather than discover a silent exclusion.

### Building membership periods

See the `src/cohorts.py` module docstring for the full method. Short version: sort each user's
transactions chronologically and label each period `renewed` (another transaction started
within 30 days of this one's expiry), `voluntary_cancel` (`is_cancel` flagged on this
transaction), `lapsed_no_renewal` (expired, not flagged, never renewed), or `censored` (too
close to the data cutoff to know yet — excluded from renewal/lapse rates everywhere downstream).

In [7]:
periods = cohorts.build_membership_periods(transactions)
members_cohort = cohorts.assign_cohort(members)

print(periods["outcome"].value_counts(normalize=True).round(3))
print(f"\n{periods['censored'].sum():,} periods ({periods['censored'].mean():.2%}) are censored.")

outcome
renewed             0.867
lapsed_no_renewal   0.055
censored            0.053
voluntary_cancel    0.024
Name: proportion, dtype: float64

1,227,117 periods (5.34%) are censored.


### Cache intermediate tables
`data/` is gitignored — this is a local speed cache. Delete it and rerun this notebook to rebuild.

In [8]:
interim_dir = Path("../data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)

periods.to_parquet(interim_dir / "periods.parquet")
members_cohort.to_parquet(interim_dir / "members_cohort.parquet")
train.to_parquet(interim_dir / "train.parquet")

print("Cached:", [p.name for p in interim_dir.glob("*.parquet")])

Cached: ['members_cohort.parquet', 'periods.parquet', 'train.parquet']


### Summary

- Loaded the three approved files only; row counts and coverage printed above.
- `bd` has known data-quality issues — flagged via `age_valid`, never silently cleaned.
- Membership periods reconstructed with an explicit right-censoring rule so recent activity
  doesn't get miscounted as churn just because there hasn't been time to observe a renewal yet.
- Every later notebook loads `periods.parquet` / `members_cohort.parquet` from this cache
  rather than rebuilding from raw transactions.